# TLC FHVHV pre/post transformations report

Data source: https://www.nyc.gov/site/tlc/about/tlc-trip-record-data.page

**Note**: Data for December 2025 is not available yet.

This notebook analyze the raw format of FHVHV in search of outliers, nulls value.

After running `src/transformations/fhvhvTransformations.py` we get a cleaner and richer dataset without nulls, outliers and new faetures.

A report comparison between both dataset is down below the notebook.

### Field Descriptions

| Field Name | Description | Relevance | Data Type |
|------------|-------------|-----------|-----------|
| **hvfhs_license_num** | The TLC license number of the HVFHS base or business.<br><br>As of September 2019, the HVFHS licensees are:<br>• **HV0002**: Juno<br>• **HV0003**: Uber<br>• **HV0004**: Via<br>• **HV0005**: Lyft | Medium | String |
| **dispatching_base_num** | The TLC Base License Number of the base that dispatched the trip. | Low | String |
| **originating_base_num** | Base number of the base that received the original trip request. | Low | String |
| **request_datetime** | Date/time when passenger requested to be picked up. | High | Datetime |
| **on_scene_datetime** | Date/time when driver arrived at the pick-up location (Accessible Vehicles-only). | Low | Datetime |
| **pickup_datetime** | The date and time of the trip pick-up. | High | Datetime |
| **dropoff_datetime** | The date and time of the trip drop-off. | High | Datetime |
| **PULocationID** | TLC Taxi Zone in which the trip began. | High | Int64 |
| **DOLocationID** | TLC Taxi Zone in which the trip ended. | High | Int64 |
| **trip_miles** | Total miles for passenger trip. | High | Float64 |
| **trip_time** | Total time in seconds for passenger trip. | High | Int64 |
| **base_passenger_fare** | Base passenger fare before tolls, tips, taxes, and fees. | High | Float64 |
| **tolls** | Total amount of all tolls paid in trip. | Medium | Float64 |
| **bcf** | Total amount collected in trip for Black Car Fund. | Low | Float64 |
| **sales_tax** | Total amount collected in trip for NYS sales tax. | Medium | Float64 |
| **congestion_surcharge** | Total amount collected in trip for NYS congestion surcharge. | High | Float64 |
| **airport_fee** | $2.50 for both drop off and pick up at LaGuardia, Newark, and John F. Kennedy airports. | Medium | Float64 |
| **tips** | Total amount of tips received from passenger. | Medium | Float64 |
| **driver_pay** | Total driver pay (not including tolls or tips and net of commission, surcharges, or taxes). | High | Float64 |
| **shared_request_flag** | Did the passenger agree to a shared/pooled ride, regardless of whether they were matched? (Y/N) | Medium | String |
| **shared_match_flag** | Did the passenger share the vehicle with another passenger who booked separately at any point during the trip? (Y/N) | Medium | String |
| **access_a_ride_flag** | Was the trip administered on behalf of the Metropolitan Transportation Authority (MTA)? (Y/N) | Low | String |
| **wav_request_flag** | Did the passenger request a wheelchair-accessible vehicle (WAV)? (Y/N) | Low | String |
| **wav_match_flag** | Did the trip occur in a wheelchair-accessible vehicle (WAV)? (Y/N) | Low | String |
| **cbd_congestion_fee** | Per-trip charge for MTA's Congestion Relief Zone starting Jan. 5, 2025. | High | Float64 |

**Note:** `cbd_congestion_fee` was introduced in 2025, previous years don't have this field.

`dispatching_base_num`, `originating_base_num`, `wav_request_flag`, `wav_match_flag`, `access_a_ride_flag`, `bcf` and `on_scene_datetime` are low relevance so will be excluded from the final dataset.

## Libraries & Config

In [1]:
import findspark
findspark.init()
import pyspark  
from pyspark.sql import SparkSession, DataFrame
from pyspark.sql.types import (
    StructType, StructField, StringType, TimestampNTZType,
    IntegerType, DoubleType, LongType
)
from pyspark.sql import functions as f
from pathlib import Path

In [2]:
projectRoot = Path.cwd().parents[2]

ZONE_CSV = Path(projectRoot)/ "data" / "raw" / "tlc_trip_record" / "taxi_zone_lookup.csv"
FHVHV_PARQUETS = Path(projectRoot) / "data" / "raw" / "tlc_trip_record" / "fhvhv"
OUTPUT = Path(projectRoot) / "data" / "clean" / "clean_tlc_trip_record" / "fhvhv"

MIN_DATETIME = "2020-01-01"
MAX_DATETIME =  "2026-01-01"

MAX_TRIP_MILES = 60         # 60 miles  = 96,50 km
MIN_TRIP_MILES = 0.1        # 0,1 miles = 160 meters
MAX_TRIP_TIME = 10_800      # 10_800 seconds = 3 hours
MIN_TRIP_TIME = 60          # 60 seconds = 1 min

MIN_FARE = 1.0
MAX_FARE = 500.00           # Base passenger fare (before fees, taxes, etc)

UNKNOWN_LOCATIONS = [264, 265]

## Init Spark

In [ ]:
# NOTE:
# Change the configs according to your device if you wanna run it.
# If we want to run this code on the cloud,
# we will need to change the configs too.

def stop(spark):
    spark.sparkContext.stop()

spark = (SparkSession.builder
        .master("local[12]")
        .config("spark.driver.memory", "24g")
        .config("spark.executor.memory", "24g")
        .config("spark.sql.shuffle.partitions", "200")
        .config("spark.sql.adaptive.advisoryPartitionSizeInBytes", "128m")
        .config("spark.driver.maxResultSize", "2g")
        .config("spark.sql.session.timeZone", "America/New_York")
        .config("spark.sql.timestampType", "TIMESTAMP_NTZ")
        # Network binding fixes for macOS
        .config("spark.driver.host", "127.0.0.1")
        .config("spark.driver.bindAddress", "127.0.0.1")
        .getOrCreate())

In [ ]:
spark.sparkContext

# Raw Data

## Load raw data

In [4]:
def load_fhvhv(spark: SparkSession) -> DataFrame:

    schema = StructType([
        StructField("hvfhs_license_num",        StringType()),
        StructField("dispatching_base_num",     StringType()),
        StructField("originating_base_num",     StringType()),
        StructField("request_datetime",         TimestampNTZType()),
        StructField("on_scene_datetime",        TimestampNTZType()),
        StructField("pickup_datetime",          TimestampNTZType()),
        StructField("dropoff_datetime",         TimestampNTZType()),
        StructField("PULocationID",             LongType()),
        StructField("DOLocationID",             LongType()),
        StructField("trip_miles",               DoubleType()),
        StructField("trip_time",                LongType()),
        StructField("base_passenger_fare",      DoubleType()),
        StructField("tolls",                    DoubleType()),
        StructField("bcf",                      DoubleType()),
        StructField("sales_tax",                DoubleType()),
        StructField("congestion_surcharge",     DoubleType()),
        StructField("airport_fee",              DoubleType()),
        StructField("tips",                     DoubleType()),
        StructField("driver_pay",               DoubleType()),
        StructField("shared_request_flag",      StringType()),
        StructField("shared_match_flag",        StringType()),
        StructField("access_a_ride_flag",       StringType()),
        StructField("wav_request_flag",         StringType()),
        StructField("wav_match_flag",           StringType()),
        StructField("cbd_congestion_fee",       DoubleType()),
    ])

    path = str(FHVHV_PARQUETS / "*.parquet")
    df = spark.read.schema(schema).parquet(path)
    return df

def load_zones(spark: SparkSession) -> DataFrame:
    path = str(ZONE_CSV)
    df = spark.read.option("header", "true").csv(path)
    df = df.select(
        f.col("LocationID").cast(IntegerType()).alias("LocationID"),
        f.col("Borough").alias("Borough"),
        f.col("Zone").alias("Zone"),
        f.col("service_zone").alias("service_zone"),
    )
    return df

In [ ]:
raw_data = load_fhvhv(spark)

In [6]:
zones = load_zones(spark)

## Stats, Nulls, Outliers

In [ ]:
raw_data.select(
    f.percentile_approx("base_passenger_fare", [0.95, 0.99, 0.999, 0.9999])
    .alias("percentiles")
).show(truncate=False)

# +--------------------------------+
# |percentiles                     |
# +--------------------------------+
# |[60.39, 105.84, 199.91, 8157.74]|
# +--------------------------------+


In [ ]:
raw_data.select(
    f.percentile_approx("trip_miles", [0.95, 0.99, 0.999, 0.9999])
    .alias("percentiles")
).show(truncate=False)

# +------------------------------+
# |percentiles                   |
# +------------------------------+
# |[16.09, 26.32, 52.18, 5380.78]|
# +------------------------------+

In [ ]:
raw_data.select(
    f.percentile_approx("trip_time", [0.95, 0.99, 0.999, 0.9999])
    .alias("percentiles")
).show(truncate=False)

# +--------------------------+
# |percentiles               |
# +--------------------------+
# |[2741, 4107, 6293, 240764]|
# +--------------------------+

In [ ]:
raw_data.select(
    f.percentile_approx("tips", [0.95, 0.99, 0.999, 0.9999])
    .alias("percentiles")
).show(truncate=False)

# +---------------------------+
# |percentiles                |
# +---------------------------+
# |[6.27, 14.88, 30.7, 1000.0]|
# +---------------------------+

### Zones


In [8]:
zones.printSchema()

root
 |-- LocationID: integer (nullable = true)
 |-- Borough: string (nullable = true)
 |-- Zone: string (nullable = true)
 |-- service_zone: string (nullable = true)



In [9]:
zones.show(3)

+----------+-------+--------------------+------------+
|LocationID|Borough|                Zone|service_zone|
+----------+-------+--------------------+------------+
|         1|    EWR|      Newark Airport|         EWR|
|         2| Queens|         Jamaica Bay|   Boro Zone|
|         3|  Bronx|Allerton/Pelham G...|   Boro Zone|
+----------+-------+--------------------+------------+
only showing top 3 rows


### Raw Data

In [ ]:
total_rows = raw_data.count()
total_rows  #1.223.764.320

In [11]:
num_cols = ["trip_miles", "trip_time", "base_passenger_fare", "tolls",
            "sales_tax", "congestion_surcharge", "airport_fee", 
            "tips","driver_pay", "cbd_congestion_fee"]

Summary

In [ ]:
raw_summary = raw_data.select(num_cols).summary().toPandas()

In [13]:
raw_summary

,summary,trip_miles,trip_time,base_passenger_fare,tolls,sales_tax,congestion_surcharge,airport_fee,tips,driver_pay,cbd_congestion_fee
0,count,1223764320,1223764320,1223764320,1223764320,1223764320,1223764320,1043959242,1223764320,1223764320,221481246
1,mean,4.973223340084099,1157.063595499336,23.97853514491241,1.0650244777456956,2.0196488596393274,1.0318176205284364,0.19864306798291645,1.0305199480728382,18.832360728867663,0.5124979069333934
2,stddev,5.761256987473091,828.8557356920786,20.983326855347066,3.7406433583477576,1.732912974186701,1.3294886876206802,0.6795231620829909,3.136825101804839,16.080452670588986,0.7114041086022336
3,min,0.0,0,-520.11,0.0,0.0,0.0,0.0,0.0,-2035.92,0.0
4,25%,1.59,590,11.43,0.0,0.95,0.0,0.0,0.0,8.57,0.0
5,50%,2.99,941,17.95,0.0,1.52,0.0,0.0,0.0,14.18,0.0
6,75%,6.18,1480,29.0,0.0,2.5,2.75,0.0,0.0,23.59,1.5
7,max,5380.78,240764,8157.74,504.58,724.08,13.75,10.0,1000.0,4894.62,4.5


Count Nulls

In [14]:
key_cols = [
    "hvfhs_license_num", "request_datetime", "pickup_datetime", "dropoff_datetime",
    "PULocationID", "DOLocationID", "trip_miles", "trip_time",
    "base_passenger_fare", "tolls", "sales_tax", "congestion_surcharge",
    "airport_fee", "tips", "driver_pay", "shared_request_flag","shared_match_flag","cbd_congestion_fee"
]

In [15]:
def count_nulls(df:DataFrame, key_cols:list[str]):
    null_counts = {}
    for col in key_cols:
        if col in df.columns:
            nulls = df.where(f.col(col).isNull()).count()
            null_counts[col] = nulls

    for col, n in null_counts.items():
        percent = (n / total_rows * 100) if total_rows > 0 else 0
        print(f"{col}: {n:,} ({percent:.2f}%)")

In [ ]:
count_nulls(raw_data, key_cols)
# request_datetime: 31 (0.00%)
# airport_fee: 179,805,078 (14.69%)
# cbd_congestion_fee: 1,002,283,074 (81.90%)

- `request_datetime`: Just delete the 31 nulls.

- `Airport_fee`: Two conditions
    
    1. Delete the observations where any locationID (PU or DO) is from an airport [1, 132, 138]
    
    2. If the locationID is not an airport replace it with zero.

- `cbd_congestion_fee` : Is showhing a big number of nulls. This is because this fee was introduced in `2025`, pre 2025 will be nulls. If there are nulls in 2025 observations we can replace it with zero.

In [17]:
airport_ids = [1, 132, 138]

aiport_cond_one = raw_data.where(
    ((f.col("DOLocationID").isin(airport_ids)) | (f.col("PULocationID").isin(airport_ids))) 
    & (f.col("airport_fee").isNull())
).count()

aiport_cond_two = raw_data.where(
    ~((f.col("DOLocationID").isin(airport_ids)) | (f.col("PULocationID").isin(airport_ids))) 
    & (f.col("airport_fee").isNull())
).count()

Number of obeservations that satisfy condition 1, these will be deleted

In [ ]:
aiport_cond_one # 7_194_433

Number of obeservations that satisfy condition 2, these will be filled w/ 0

In [ ]:
aiport_cond_two # 172_610_645

Check

In [ ]:
aiport_cond_one + aiport_cond_two # 179,805,078

`LocationID`: Decided to delete the rows where any locationID is 264 or 265 which are either "unknown" or "N/A" (Outside of NYC)

Negative numbers

They are `base_passenger_fare` & `driver_pay` from summary

In [ ]:
with_negative = ["base_passenger_fare", "driver_pay"]

negative_counts = raw_data.select([
    f.sum(f.when(f.col(col) < 0, 1).otherwise(0)).alias(f"{col}")
    for col in with_negative
]).toPandas()

negative_counts
# base_passenger_fare 697.545
# driver_pay 21.155

# FINAL DATA

The dataset obtained after running `src/transformations/fhvhv_clean_enrich.py`

In [22]:
final_data = spark.read.parquet(str(OUTPUT))

In [ ]:
final_data.printSchema()

In [ ]:
final_data_rows = final_data.count()
total_rows - final_data_rows    # Dropped 132.505.649 rows

In [ ]:
count_nulls(final_data, final_data.columns)
# 0 nulls in all columns

In [27]:
num_cols = ["total_amount", "wait_time","company_fees"]

final_summary = final_data.select(num_cols).summary().toPandas()

In [28]:
final_summary

,summary,total_amount,wait_time,company_fees
0,count,1091258671,1091258671,1091258671
1,mean,27.51414165691975,294.96074560125993,10.329502236461183
2,stddev,21.225036100229733,210.55793851773817,10.625855153286434
3,min,1.0,1,0.009999999999990905
4,25%,13.6,174,4.07
5,50%,21.19,250,7.130000000000001
6,75%,34.03,363,12.72
7,max,810.42,313031,690.68
